# Phase 3c: Ensemble Models
## DNA Gene Mapping Project
**Author:** Sharique Mohammad  
**Date:** February 2026  

---

## Objective
Train advanced ensemble models with hyperparameter tuning:
- **Random Forest**
- **XGBoost**
- **LightGBM**

## Goal
Beat baseline performance:
- Variant F1 > 0.78 (baseline: 0.78)
- SV Recall > 0.99 (baseline: 1.0)

## Strategy
- Randomized search for hyperparameters
- 3-fold cross-validation
- Feature importance comparison

---
## 1. Setup

In [ ]:
# Imports
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
import json
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve,
    f1_score, precision_score, recall_score, accuracy_score,
    make_scorer
)
import xgboost as xgb
import lightgbm as lgb

import warnings
warnings.filterwarnings('ignore')

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print(" Imports successful")

In [ ]:
# Configuration
PROJECT_ROOT = Path.cwd().parent.parent
DATA_DIR = PROJECT_ROOT / "data" / "ml"
MODEL_DIR = PROJECT_ROOT / "models"
METRICS_DIR = PROJECT_ROOT / "data" / "ml" / "metrics"
FIGURES_DIR = PROJECT_ROOT / "data" / "analytical" / "figures" / "phase3"

RANDOM_STATE = 42
N_ITER = 20  # Randomized search iterations
CV_FOLDS = 3  # Cross-validation folds
N_JOBS = -1   # Use all CPU cores

print("="*80)
print("PHASE 3C: ENSEMBLE MODELS WITH HYPERPARAMETER TUNING")
print("="*80)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Random state: {RANDOM_STATE}")
print(f"Tuning iterations: {N_ITER}")
print(f"CV folds: {CV_FOLDS}")
print("="*80)

---
## 2. Load Prepared Datasets

In [ ]:
print("Loading prepared datasets...")

# Load variant pathogenicity datasets
with open(DATA_DIR / "variant_train_balanced.pkl", 'rb') as f:
    train_bal = pickle.load(f)
    X_train, y_train = train_bal['X'], train_bal['y']

with open(DATA_DIR / "variant_validation.pkl", 'rb') as f:
    val_data = pickle.load(f)
    X_val, y_val = val_data['X'], val_data['y']

with open(DATA_DIR / "variant_test.pkl", 'rb') as f:
    test_data = pickle.load(f)
    X_test, y_test = test_data['X'], test_data['y']

print("\nVariant Pathogenicity:")
print(f"  Train: {X_train.shape}")
print(f"  Validation: {X_val.shape}")
print(f"  Test: {X_test.shape}")

# Load SV datasets
with open(DATA_DIR / "sv_train.pkl", 'rb') as f:
    sv_train = pickle.load(f)
    X_sv_train, y_sv_train = sv_train['X'], sv_train['y']

with open(DATA_DIR / "sv_validation.pkl", 'rb') as f:
    sv_val = pickle.load(f)
    X_sv_val, y_sv_val = sv_val['X'], sv_val['y']

with open(DATA_DIR / "sv_test.pkl", 'rb') as f:
    sv_test = pickle.load(f)
    X_sv_test, y_sv_test = sv_test['X'], sv_test['y']

print("\nStructural Variants:")
print(f"  Train: {X_sv_train.shape}")
print(f"  Validation: {X_sv_val.shape}")
print(f"  Test: {X_sv_test.shape}")

---
## 3. Hyperparameter Grids

In [ ]:
# Random Forest parameter grid
rf_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2'],
    'bootstrap': [True, False]
}

# XGBoost parameter grid
xgb_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'gamma': [0, 0.1, 0.2]
}

# LightGBM parameter grid
lgb_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'num_leaves': [31, 50, 70],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

print("Hyperparameter grids defined:")
print(f"  Random Forest: {len(rf_params)} parameters")
print(f"  XGBoost: {len(xgb_params)} parameters")
print(f"  LightGBM: {len(lgb_params)} parameters")

---
## 4. Helper Functions

In [ ]:
def tune_and_evaluate(model, param_grid, X_train, y_train, X_val, y_val, 
                     model_name, task_name, scoring='f1'):
    """
    Hyperparameter tuning with RandomizedSearchCV.
    
    Returns:
        best_model, results_dict, y_val_pred, y_val_proba
    """
    print(f"\n{model_name} - {task_name}")
    print("-" * 80)
    
    # Randomized search
    print(f"Hyperparameter tuning ({N_ITER} iterations, {CV_FOLDS}-fold CV)...")
    start_time = time.time()
    
    search = RandomizedSearchCV(
        model,
        param_distributions=param_grid,
        n_iter=N_ITER,
        cv=CV_FOLDS,
        scoring=scoring,
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE,
        verbose=0
    )
    
    search.fit(X_train, y_train)
    elapsed = time.time() - start_time
    
    print(f" Tuning complete ({elapsed:.1f}s)")
    print(f"Best CV score: {search.best_score_:.4f}")
    print(f"\nBest parameters:")
    for param, value in search.best_params_.items():
        print(f"  {param}: {value}")
    
    # Evaluate on validation set
    best_model = search.best_estimator_
    y_train_pred = best_model.predict(X_train)
    y_val_pred = best_model.predict(X_val)
    y_val_proba = best_model.predict_proba(X_val)[:, 1]
    
    results = {
        'model': model_name,
        'task': task_name,
        'best_params': search.best_params_,
        'cv_score': search.best_score_,
        'train': {
            'accuracy': accuracy_score(y_train, y_train_pred),
            'precision': precision_score(y_train, y_train_pred),
            'recall': recall_score(y_train, y_train_pred),
            'f1': f1_score(y_train, y_train_pred)
        },
        'validation': {
            'accuracy': accuracy_score(y_val, y_val_pred),
            'precision': precision_score(y_val, y_val_pred),
            'recall': recall_score(y_val, y_val_pred),
            'f1': f1_score(y_val, y_val_pred),
            'roc_auc': roc_auc_score(y_val, y_val_proba)
        }
    }
    
    print(f"\nValidation Performance:")
    print(f"  F1:        {results['validation']['f1']:.4f}")
    print(f"  Precision: {results['validation']['precision']:.4f}")
    print(f"  Recall:    {results['validation']['recall']:.4f}")
    print(f"  ROC-AUC:   {results['validation']['roc_auc']:.4f}")
    
    return best_model, results, y_val_pred, y_val_proba

def plot_confusion_matrix(y_true, y_pred, model_name, task_name, save_path):
    """Plot confusion matrix"""
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(f'{model_name} - {task_name}\nConfusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()

def plot_feature_importance(model, feature_names, model_name, task_name, save_path, top_n=20):
    """Plot feature importance"""
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
    else:
        print(f"  No feature importance for {model_name}")
        return None
    
    # Get top features
    indices = np.argsort(importances)[-top_n:]
    top_features = [feature_names[i] for i in indices]
    top_importances = importances[indices]
    
    # Plot
    plt.figure(figsize=(10, 8))
    plt.barh(range(len(top_features)), top_importances)
    plt.yticks(range(len(top_features)), top_features)
    plt.xlabel('Importance')
    plt.title(f'{model_name} - {task_name}\nTop {top_n} Features')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    
    return list(zip(top_features, top_importances))

print(" Helper functions defined")

---
## 5. Task 1: Variant Pathogenicity - Random Forest

In [ ]:
print("="*80)
print("TASK 1: VARIANT PATHOGENICITY PREDICTION")
print("="*80)

# Random Forest
rf_model, rf_results, rf_pred, rf_proba = tune_and_evaluate(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=N_JOBS),
    rf_params, X_train, y_train, X_val, y_val,
    "Random Forest", "Variant Pathogenicity"
)

# Save model
rf_file = MODEL_DIR / "ensemble_rf_variants.pkl"
with open(rf_file, 'wb') as f:
    pickle.dump(rf_model, f)
print(f"\n Model saved: {rf_file.name}")

In [ ]:
# Visualizations
print("\nGenerating visualizations...")

plot_confusion_matrix(
    y_val, rf_pred, "Random Forest", "Variant Pathogenicity",
    FIGURES_DIR / "13_rf_variants_confusion_matrix.png"
)

rf_top_features = plot_feature_importance(
    rf_model, X_train.columns, "Random Forest", "Variant Pathogenicity",
    FIGURES_DIR / "14_rf_variants_feature_importance.png"
)

print(" Visualizations saved")
if rf_top_features:
    print("\nTop 5 Features:")
    for feat, imp in rf_top_features[-5:][::-1]:
        print(f"  {feat}: {imp:.4f}")

---
## 6. Task 1: Variant Pathogenicity - XGBoost

In [ ]:
# XGBoost
xgb_model, xgb_results, xgb_pred, xgb_proba = tune_and_evaluate(
    xgb.XGBClassifier(random_state=RANDOM_STATE, n_jobs=N_JOBS, eval_metric='logloss'),
    xgb_params, X_train, y_train, X_val, y_val,
    "XGBoost", "Variant Pathogenicity"
)

# Save model
xgb_file = MODEL_DIR / "ensemble_xgb_variants.pkl"
with open(xgb_file, 'wb') as f:
    pickle.dump(xgb_model, f)
print(f"\nModel saved: {xgb_file.name}")

In [ ]:
# Visualizations
print("\nGenerating visualizations...")

plot_confusion_matrix(
    y_val, xgb_pred, "XGBoost", "Variant Pathogenicity",
    FIGURES_DIR / "15_xgb_variants_confusion_matrix.png"
)

xgb_top_features = plot_feature_importance(
    xgb_model, X_train.columns, "XGBoost", "Variant Pathogenicity",
    FIGURES_DIR / "16_xgb_variants_feature_importance.png"
)

print(" Visualizations saved")
if xgb_top_features:
    print("\nTop 5 Features:")
    for feat, imp in xgb_top_features[-5:][::-1]:
        print(f"  {feat}: {imp:.4f}")

---
## 7. Task 1: Variant Pathogenicity - LightGBM

In [ ]:
# LightGBM
lgb_model, lgb_results, lgb_pred, lgb_proba = tune_and_evaluate(
    lgb.LGBMClassifier(random_state=RANDOM_STATE, n_jobs=N_JOBS, verbose=-1),
    lgb_params, X_train, y_train, X_val, y_val,
    "LightGBM", "Variant Pathogenicity"
)

# Save model
lgb_file = MODEL_DIR / "ensemble_lgb_variants.pkl"
with open(lgb_file, 'wb') as f:
    pickle.dump(lgb_model, f)
print(f"\n Model saved: {lgb_file.name}")

In [ ]:
# Visualizations
print("\nGenerating visualizations...")

plot_confusion_matrix(
    y_val, lgb_pred, "LightGBM", "Variant Pathogenicity",
    FIGURES_DIR / "17_lgb_variants_confusion_matrix.png"
)

lgb_top_features = plot_feature_importance(
    lgb_model, X_train.columns, "LightGBM", "Variant Pathogenicity",
    FIGURES_DIR / "18_lgb_variants_feature_importance.png"
)

print(" Visualizations saved")
if lgb_top_features:
    print("\nTop 5 Features:")
    for feat, imp in lgb_top_features[-5:][::-1]:
        print(f"  {feat}: {imp:.4f}")

---
## 8. Task 2: SV Risk - Random Forest

In [ ]:
print("\n" + "="*80)
print("TASK 2: STRUCTURAL VARIANT RISK PREDICTION")
print("="*80)

# Random Forest
rf_sv_model, rf_sv_results, rf_sv_pred, rf_sv_proba = tune_and_evaluate(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=N_JOBS),
    rf_params, X_sv_train, y_sv_train, X_sv_val, y_sv_val,
    "Random Forest", "SV Risk", scoring='recall'
)

# Save model
rf_sv_file = MODEL_DIR / "ensemble_rf_sv.pkl"
with open(rf_sv_file, 'wb') as f:
    pickle.dump(rf_sv_model, f)
print(f"\n Model saved: {rf_sv_file.name}")

In [ ]:
# Visualizations
print("\nGenerating visualizations...")

plot_confusion_matrix(
    y_sv_val, rf_sv_pred, "Random Forest", "SV Risk",
    FIGURES_DIR / "19_rf_sv_confusion_matrix.png"
)

rf_sv_top_features = plot_feature_importance(
    rf_sv_model, X_sv_train.columns, "Random Forest", "SV Risk",
    FIGURES_DIR / "20_rf_sv_feature_importance.png", top_n=13
)

print(" Visualizations saved")

---
## 9. Task 2: SV Risk - XGBoost

In [ ]:
# XGBoost
xgb_sv_model, xgb_sv_results, xgb_sv_pred, xgb_sv_proba = tune_and_evaluate(
    xgb.XGBClassifier(random_state=RANDOM_STATE, n_jobs=N_JOBS, eval_metric='logloss'),
    xgb_params, X_sv_train, y_sv_train, X_sv_val, y_sv_val,
    "XGBoost", "SV Risk", scoring='recall'
)

# Save model
xgb_sv_file = MODEL_DIR / "ensemble_xgb_sv.pkl"
with open(xgb_sv_file, 'wb') as f:
    pickle.dump(xgb_sv_model, f)
print(f"\n Model saved: {xgb_sv_file.name}")

In [ ]:
# Visualizations
print("\nGenerating visualizations...")

plot_confusion_matrix(
    y_sv_val, xgb_sv_pred, "XGBoost", "SV Risk",
    FIGURES_DIR / "21_xgb_sv_confusion_matrix.png"
)

xgb_sv_top_features = plot_feature_importance(
    xgb_sv_model, X_sv_train.columns, "XGBoost", "SV Risk",
    FIGURES_DIR / "22_xgb_sv_feature_importance.png", top_n=13
)

print(" Visualizations saved")

---
## 10. Task 2: SV Risk - LightGBM

In [ ]:
# LightGBM
lgb_sv_model, lgb_sv_results, lgb_sv_pred, lgb_sv_proba = tune_and_evaluate(
    lgb.LGBMClassifier(random_state=RANDOM_STATE, n_jobs=N_JOBS, verbose=-1),
    lgb_params, X_sv_train, y_sv_train, X_sv_val, y_sv_val,
    "LightGBM", "SV Risk", scoring='recall'
)

# Save model
lgb_sv_file = MODEL_DIR / "ensemble_lgb_sv.pkl"
with open(lgb_sv_file, 'wb') as f:
    pickle.dump(lgb_sv_model, f)
print(f"\n Model saved: {lgb_sv_file.name}")

In [ ]:
# Visualizations
print("\nGenerating visualizations...")

plot_confusion_matrix(
    y_sv_val, lgb_sv_pred, "LightGBM", "SV Risk",
    FIGURES_DIR / "23_lgb_sv_confusion_matrix.png"
)

lgb_sv_top_features = plot_feature_importance(
    lgb_sv_model, X_sv_train.columns, "LightGBM", "SV Risk",
    FIGURES_DIR / "24_lgb_sv_feature_importance.png", top_n=13
)

print(" Visualizations saved")

---
## 11. Model Comparison

In [ ]:
# Create comparison table
comparison = pd.DataFrame([
    {
        'Task': 'Variant Pathogenicity',
        'Model': 'Random Forest',
        'F1': rf_results['validation']['f1'],
        'Precision': rf_results['validation']['precision'],
        'Recall': rf_results['validation']['recall'],
        'ROC-AUC': rf_results['validation']['roc_auc'],
        'CV Score': rf_results['cv_score']
    },
    {
        'Task': 'Variant Pathogenicity',
        'Model': 'XGBoost',
        'F1': xgb_results['validation']['f1'],
        'Precision': xgb_results['validation']['precision'],
        'Recall': xgb_results['validation']['recall'],
        'ROC-AUC': xgb_results['validation']['roc_auc'],
        'CV Score': xgb_results['cv_score']
    },
    {
        'Task': 'Variant Pathogenicity',
        'Model': 'LightGBM',
        'F1': lgb_results['validation']['f1'],
        'Precision': lgb_results['validation']['precision'],
        'Recall': lgb_results['validation']['recall'],
        'ROC-AUC': lgb_results['validation']['roc_auc'],
        'CV Score': lgb_results['cv_score']
    },
    {
        'Task': 'SV Risk',
        'Model': 'Random Forest',
        'F1': rf_sv_results['validation']['f1'],
        'Precision': rf_sv_results['validation']['precision'],
        'Recall': rf_sv_results['validation']['recall'],
        'ROC-AUC': rf_sv_results['validation']['roc_auc'],
        'CV Score': rf_sv_results['cv_score']
    },
    {
        'Task': 'SV Risk',
        'Model': 'XGBoost',
        'F1': xgb_sv_results['validation']['f1'],
        'Precision': xgb_sv_results['validation']['precision'],
        'Recall': xgb_sv_results['validation']['recall'],
        'ROC-AUC': xgb_sv_results['validation']['roc_auc'],
        'CV Score': xgb_sv_results['cv_score']
    },
    {
        'Task': 'SV Risk',
        'Model': 'LightGBM',
        'F1': lgb_sv_results['validation']['f1'],
        'Precision': lgb_sv_results['validation']['precision'],
        'Recall': lgb_sv_results['validation']['recall'],
        'ROC-AUC': lgb_sv_results['validation']['roc_auc'],
        'CV Score': lgb_sv_results['cv_score']
    }
])

print("\n" + "="*80)
print("ENSEMBLE MODEL COMPARISON")
print("="*80)
print(comparison.to_string(index=False))

# Save comparison
comparison.to_csv(METRICS_DIR / "ensemble_model_comparison.csv", index=False)
print(f"\n Comparison saved: ensemble_model_comparison.csv")

---
## 12. Baseline vs Ensemble Comparison

In [ ]:
# Load baseline results
with open(METRICS_DIR / "baseline_summary.json", 'r') as f:
    baseline = json.load(f)

# Compare variants
print("\n" + "="*80)
print("BASELINE VS ENSEMBLE - VARIANT PATHOGENICITY")
print("="*80)

baseline_f1 = baseline['variant_pathogenicity']['best_f1']
best_ensemble_f1 = max(rf_results['validation']['f1'], 
                       xgb_results['validation']['f1'],
                       lgb_results['validation']['f1'])

improvement = ((best_ensemble_f1 - baseline_f1) / baseline_f1) * 100

print(f"Baseline (Decision Tree): F1 = {baseline_f1:.4f}")
print(f"Best Ensemble:            F1 = {best_ensemble_f1:.4f}")
print(f"Improvement:              {improvement:+.2f}%")

if best_ensemble_f1 > baseline_f1:
    print("\n Ensemble models beat baseline!")
else:
    print("\nWARNING: Ensemble did not improve over baseline")

# Compare SVs
print("\n" + "="*80)
print("BASELINE VS ENSEMBLE - SV RISK")
print("="*80)

baseline_recall = baseline['sv_risk']['best_recall']
best_ensemble_recall = max(rf_sv_results['validation']['recall'],
                          xgb_sv_results['validation']['recall'],
                          lgb_sv_results['validation']['recall'])

print(f"Baseline (Decision Tree): Recall = {baseline_recall:.4f}")
print(f"Best Ensemble:            Recall = {best_ensemble_recall:.4f}")

if best_ensemble_recall >= 0.999:
    print("\nNOTE: Near-perfect performance suggests possible data leakage")

---
## 13. Summary Report

In [ ]:
# Compile summary
summary = {
    'timestamp': datetime.now().isoformat(),
    'models_trained': 6,
    'variant_pathogenicity': {
        'random_forest': rf_results,
        'xgboost': xgb_results,
        'lightgbm': lgb_results,
        'best_model': comparison[comparison['Task'] == 'Variant Pathogenicity'].sort_values('F1', ascending=False).iloc[0]['Model'],
        'best_f1': best_ensemble_f1,
        'baseline_f1': baseline_f1,
        'improvement_pct': improvement
    },
    'sv_risk': {
        'random_forest': rf_sv_results,
        'xgboost': xgb_sv_results,
        'lightgbm': lgb_sv_results,
        'best_model': comparison[comparison['Task'] == 'SV Risk'].sort_values('Recall', ascending=False).iloc[0]['Model'],
        'best_recall': best_ensemble_recall
    }
}

# Save summary
summary_file = METRICS_DIR / "ensemble_summary.json"
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2)

print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"\nVariant Pathogenicity:")
print(f"  Best Model: {summary['variant_pathogenicity']['best_model']}")
print(f"  Best F1: {summary['variant_pathogenicity']['best_f1']:.4f}")
print(f"  Improvement over baseline: {summary['variant_pathogenicity']['improvement_pct']:.2f}%")
print(f"\nSV Risk:")
print(f"  Best Model: {summary['sv_risk']['best_model']}")
print(f"  Best Recall: {summary['sv_risk']['best_recall']:.4f}")

print(f"\n Summary saved: {summary_file.name}")

In [ ]:
# List all outputs
print("\n" + "="*80)
print("FILES CREATED")
print("="*80)

print(f"\nModels:")
for file in sorted(MODEL_DIR.glob("ensemble_*.pkl")):
    print(f"  - {file.name}")

print(f"\nMetrics:")
for file in sorted(METRICS_DIR.glob("ensemble_*")):
    print(f"  - {file.name}")

print(f"\nFigures (new):")
for file in sorted(FIGURES_DIR.glob("1[3-9]*.png")) + sorted(FIGURES_DIR.glob("2*.png")):
    print(f"  - {file.name}")

print("\n" + "="*80)
print("PHASE 3C COMPLETE: ENSEMBLE MODELS TRAINED")
print("="*80)
print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\nNext step: 03d_model_evaluation.ipynb")
print("  - Compare all models (baseline + ensemble)")
print("  - Select best model for each task")
print("  - Evaluate on test set")
print("="*80)